# RadQA

## Preprocessing (done)

In [ ]:
!pip install -q datasets rouge_score

## Install

In [ ]:
!pip install -q openai
!pip install -q evaluate
!pip install -q transformers accelerate

In [ ]:
import pandas as pd
import os
import openai
import concurrent.futures
from tqdm import tqdm
import time
import evaluate
import torch
# import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# !ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Prompt

In [ ]:
# # v1
# messages=[
#               {"role": "system", "content": "You are a radiologist. Your task is to determine whether a given context sentence from radiology reports contains the answer to a provided question. Please respond only with 'entailment' or 'not entailment' without any additional explanations. If you cannot determine, please respond with 'not entailment'."},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 1, redacted]\nQuestion: [RadQA demonstration question 1, redacted]"},
#               {"role": "assistant", "content": "entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 2, redacted]\nQuestion: [RadQA demonstration question 2, redacted]"},
#               {"role": "assistant", "content": "not entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: {}\nQuestion: {}".format(sentence, question)}
#           ]

# # v2
# messages=[
#               {"role": "system", "content": "You are a radiologist. Your task is to determine whether a given context sentence from radiology reports contains the answer to a provided question. Please respond *only* with 'entailment' or 'not entailment' without any additional explanations."},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 1, redacted]\nQuestion: [RadQA demonstration question 1, redacted]"},
#               {"role": "assistant", "content": "entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 2, redacted]\nQuestion: [RadQA demonstration question 2, redacted]"},
#               {"role": "assistant", "content": "not entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: {}\nQuestion: {}".format(sentence, question)}
#           ]

# # v3
# messages=[
#               {"role": "system", "content": "You are a radiologist. Your task is to determine whether a given context sentence from radiology reports contains the answer to a provided question. Please respond only with 'entailment' or 'not entailment' without any additional explanations."},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 1, redacted]\nQuestion: [RadQA demonstration question 1, redacted]"},
#               {"role": "assistant", "content": "entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 2, redacted]\nQuestion: [RadQA demonstration question 2, redacted]"},
#               {"role": "assistant", "content": "not entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 3, redacted]\nQuestion: [RadQA demonstration question 3, redacted]"},
#               {"role": "assistant", "content": "entailment"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: {}\nQuestion: {}".format(sentence, question)}
#           ]

# # v5
# messages=[
#               {"role": "system", "content": "You are a radiologist. Your task is to determine whether a given context sentence from radiology reports contains the answer to a provided question. Please respond only with 'Yes' or 'No' without any additional explanations."},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 1, redacted]\nQuestion: [RadQA demonstration question 1, redacted]"},
#               {"role": "assistant", "content": "Yes"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 2, redacted]\nQuestion: [RadQA demonstration question 2, redacted]"},
#               {"role": "assistant", "content": "No"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 3, redacted]\nQuestion: [RadQA demonstration question 3, redacted]"},
#               {"role": "assistant", "content": "Yes"},
#               {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: {}\nQuestion: {}".format(sentence, question)}
#           ]

## chatGPT

In [ ]:
def chestGPT_summary(sentence, question):
  max_tries = 3
  for try_number in range(max_tries):
    try:
      response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
              {"role": "system", "content": "You are a radiologist. Your task is to determine whether a given context sentence from radiology reports contains the answer to a provided question. Please respond only with 'entailment' or 'not entailment' without any additional explanations."},
              {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 1, redacted]\nQuestion: [RadQA demonstration question 1, redacted]"},
              {"role": "assistant", "content": "entailment"},
              {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 2, redacted]\nQuestion: [RadQA demonstration question 2, redacted]"},
              {"role": "assistant", "content": "not entailment"},
              {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: [RadQA demonstration sentence 3, redacted]\nQuestion: [RadQA demonstration question 3, redacted]"},
              {"role": "assistant", "content": "entailment"},
              {"role": "user", "content": "Determine whether the given context sentence from radiology reports contains the answer to the provided question.\n\nContext sentence: {}\nQuestion: {}".format(sentence, question)}
          ]
      )
      return response['choices'][0]['message']['content']

    # except Exception as e:
    #   if try_number == max_tries - 1:
    #     print('APIError')
    #     return '\n'
    #   else:
    #     time.sleep(0.1)
    
    except openai.error.APIError as e:
      if try_number == max_tries - 1:
        print('APIError')
        return '\n'
      else:
        time.sleep(0.1)
    except openai.error.Timeout as e:
      if try_number == max_tries - 1:
        print('Timeout')
        return '\n'
      else:
        time.sleep(0.1)
    except openai.error.APIConnectionError as e:
      if try_number == max_tries - 1:
        print('APIConnectionError')
        return '\n'
      else:
        time.sleep(0.1)
    except openai.error.RateLimitError as e:
      if try_number == max_tries - 1:
        print('RateLimitError')
        return '\n'
      else:
        time.sleep(0.1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Run

In [ ]:
import time

def process_row(args):
    time.sleep(1)
    index, row = args
    corrected_text = chestGPT_summary(row['sentence'], row['question']).replace('not ', 'not_').replace('.', '')
    return index, corrected_text

# openai.api_key = '***REMOVED***' # gpt 3.5
# openai.api_key = '***REMOVED***' # cc
openai.api_key = '***REMOVED***' # cc2

# input_file = "/content/drive/MyDrive/GPT_Data/R1/radqa_test_00.csv"
input_file = "/content/drive/MyDrive/GPT_Data/R1/radqa_test_01.csv"
# input_file = "/content/drive/MyDrive/GPT_Data/R1/radqa_test_02.csv"
# input_file = "/content/drive/MyDrive/GPT_Data/R1/radqa_test_03.csv"

df = pd.read_csv(input_file)

# parallel
# with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
with concurrent.futures.ThreadPoolExecutor() as executor:
    results = dict(tqdm(executor.map(process_row, df.iterrows()), total=len(df)))

# same order as the input
df['predict'] = df.index.map(results.get)

output_file = "/content/drive/MyDrive/GPT_Data/R1/Output/test_01_output_3.csv"
# output_file = "/content/drive/MyDrive/GPT_Data/R1/Output/test_01_output.csv"

df.to_csv(output_file, index=False)

100%|██████████| 1787/1787 [05:50<00:00,  5.10it/s]


## Eval

In [ ]:
output_file = "/content/drive/MyDrive/GPT_Data/R1/Output/test_01_output_3.csv"
predict = pd.read_csv(output_file)
predict['predict'] = predict['predict'].str.replace('', '')
valid_values = ['entailment', 'not_entailment']
predict['predict'] = predict['predict'].apply(lambda x: 'not_entailment' if x not in valid_values else x)
predict[['label', 'predict']].head(10)

,label,predict
0,not_entailment,not_entailment
1,not_entailment,not_entailment
2,not_entailment,not_entailment
3,entailment,entailment
4,not_entailment,not_entailment
5,not_entailment,not_entailment
6,not_entailment,not_entailment
7,not_entailment,not_entailment
8,not_entailment,not_entailment
9,not_entailment,entailment


In [ ]:
predict['predict'].unique()

array(['not_entailment', 'entailment'], dtype=object)

In [ ]:
predict['label'] = predict['label'].replace('not_entailment', 0).replace('entailment', 1)
predict['predict'] = predict['predict'].replace('not_entailment', 0).replace('entailment', 1)

ACC = evaluate.load("accuracy")
P = evaluate.load("precision")
R = evaluate.load("recall")
F1 = evaluate.load("f1")
acc = ACC.compute(predictions=predict['predict'], references=predict['label'])
p = P.compute(predictions=predict['predict'], references=predict['label'])
r = R.compute(predictions=predict['predict'], references=predict['label'])
f1 = F1.compute(predictions=predict['predict'], references=predict['label'])
print('ACC: {:0.4f}'.format(acc['accuracy']))
print('P: {:0.4f}'.format(p['precision']))
print('R: {:0.4f}'.format(r['recall']))
print('F1: {:0.4f}'.format(f1['f1']))

ACC: 0.7957
P: 0.6203
R: 0.5326
F1: 0.5731


## radqa_test_00

### 1 
ACC: 0.8094

P: 0.5602

R: 0.5261

F1: 0.5426

### 2
ACC: 0.8090

P: 0.5581

R: 0.5326

F1: 0.5451

### 3
ACC: 0.8094

P: 0.5586

R: 0.5391

F1: 0.5487

### 4
ACC: 0.8122

P: 0.5674

R: 0.5304

F1: 0.5483

## radqa_test_01
### 1
ACC: 0.7885

P: 0.6051

R: 0.5130

F1: 0.5553
### 2
ACC: 0.7929

P: 0.6119

R: 0.5348

F1: 0.5708
### 3
ACC: 0.7957

P: 0.6203

R: 0.5326

F1: 0.5731
### 4
ACC: 0.7963

P: 0.6212

R: 0.5348

F1: 0.5748

## radqa_test_02
### 1
ACC: 0.7001

P: 0.7634

R: 0.5261

F1: 0.6229
### 2
ACC: 0.6909

P: 0.7500

R: 0.5152

F1: 0.6108
### 3
ACC: 0.7083

P: 0.7676

R: 0.5457

F1: 0.6379
### 4
ACC: 0.6950

P: 0.7531

R: 0.5239

F1: 0.6179

## radqa_test_03
### 1
ACC: 0.6233

P: 0.8905

R: 0.5304

F1: 0.6649

### 2
ACC: 0.6156

P: 0.8885

R: 0.5196

F1: 0.6557

### 3
ACC: 0.6110

P: 0.8843

R: 0.5152

F1: 0.6511

### 4
ACC: 0.6233

P: 0.8877

R: 0.5326

F1: 0.6658

||radqa_test_00|radqa_test_01|radqa_test_02|radqa_test_03|
|--|--|--|--|--|
|1|ACC: 0.8094 P: 0.5602 R: 0.5261 F1: 0.5426|ACC: 0.7885 P: 0.6051 R: 0.5130 F1: 0.5553|ACC: 0.7001 P: 0.7634 R: 0.5261 F1: 0.6229|ACC: 0.6233 P: 0.8905 R: 0.5304 F1: 0.6649|
|2|ACC: 0.8090 P: 0.5581 R: 0.5326 F1: 0.5451|ACC: 0.7929 P: 0.6119 R: 0.5348 F1: 0.5708|ACC: 0.6909 P: 0.7500 R: 0.5152 F1: 0.6108|ACC: 0.6156 P: 0.8885 R: 0.5196 F1: 0.6557|
|3|ACC: 0.8094 P: 0.5586 R: 0.5391 F1: 0.5487|ACC: 0.7957 P: 0.6203 R: 0.5326 F1: 0.5731|ACC: 0.7083 P: 0.7676 R: 0.5457 F1: 0.6379|ACC: 0.6110 P: 0.8843 R: 0.5152 F1: 0.6511|
|4|ACC: 0.8122 P: 0.5674 R: 0.5304 F1: 0.5483|ACC: 0.7963 P: 0.6212 R: 0.5348 F1: 0.5748|ACC: 0.6950 P: 0.7531 R: 0.5239 F1: 0.6179|ACC: 0.6233 P: 0.8877 R: 0.5326 F1: 0.6658|
|Average|Acc: P: R: F1: |Acc: P: R: F1: |Acc: P: R: F1: |Acc: P: R: F1: |